# GPU 활용 임베딩 생성 (Google Colab)

이 노트북은 `generate_embeddings_v3_parallel.py`를 사용하여 가속화된 임베딩 생성을 수행합니다.
Colab의 로컬 디스크 용량 한계를 고려하여 최적화된 설정을 사용합니다.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 필수 패키지 설치

In [ ]:
!pip install duckdb pandas pyarrow torch tqdm

## 3. 프로젝트 코드 준비
구글 드라이브에 있는 프로젝트 폴더로 이동하거나 코드를 복사해 옵니다.

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os
from google.colab import userdata

# 🔑 아이콘 클릭 → Add new secret
# Name: GITHUB_TOKEN
# Value: your_personal_access_token
try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token for authentication")
except:
    print("⚠️ GITHUB_TOKEN not found. Using public clone (may have rate limits)")
    use_token = False

repo_path = '/content/stock-bot2'

# 이미 클론되어 있으면 스킵
if not os.path.exists(repo_path):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git {repo_path}
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git {repo_path}

    if os.path.exists(repo_path):
        print("✅ Repository cloned successfully!")
    else:
        print("❌ Repository cloning failed. Please check your GitHub token and repository access permissions.")
        # Early exit if cloning failed to prevent further errors
        # No need to change directory or sys path if clone failed
        print(f"📂 Current directory: {os.getcwd()}")
        sys.exit("Repository cloning failed.") # Stop execution here.

else:
    print("📁 Repository already exists, updating...")
    %cd {repo_path}
    !git fetch origin && git pull
    print("✅ Repository updated!")

# 작업 디렉토리 변경 및 Python path 추가 (성공적으로 클론되거나 업데이트된 경우에만)
%cd {repo_path}
import sys
# Ensure the project root is in the Python path
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ Repository ready!")
print(f"📂 Current directory: {os.getcwd()}")

PROJECT_PATH = '/content/stock-bot2'
%cd {PROJECT_PATH}

## 4. 데이터 로컬 복사 (속도 최적화)
구글 드라이브에서 직접 DB를 읽으면 IO 속도가 매우 느려 병렬 처리가 어렵습니다.
DB 파일을 Colab 로컬 저장소(`/content`)로 복사합니다.

In [ ]:
# 드라이브의 원본 DB 경로
DRIVE_DB_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb'
LOCAL_DB_PATH = '/content/datasets.duckdb'

if not os.path.exists(LOCAL_DB_PATH):
    print("DB 파일 복사 중...")
    !cp {DRIVE_DB_PATH} {LOCAL_DB_PATH}
    print("복사 완료.")

## 5. 임베딩 생성 실행
경로 설정 시 `--output_dir`과 `--state_file`은 구글 드라이브 경로를 지정하여 중간 진행 상황이 저장되도록 합니다.

**주의:** `generate_embeddings_v3_parallel.py` 내의 `MAX_TEMP_SIZE`가 500GB로 설정되어 있다면, Colab 환경에 맞춰 **30GB** 정도로 수정하는 것을 권장합니다.

In [ ]:
!python scripts/data/generate_embeddings_v3_parallel.py \
  --db_path "/content/datasets.duckdb" \
  --model_path "/content/drive/MyDrive/path/to/best_model.pt" \
  --output_dir "/content/drive/MyDrive/path/to/embeddings_v2" \
  --state_file "/content/drive/MyDrive/path/to/processed_stocks.txt" \
  --table_name "datasets" \
  --seq_len 120 \
  --batch_size 4096 \
  --num_workers 4 \
  --device cuda